In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
# 프로젝트 기준 경로 설정
DATA_PATH = Path("../datasets/cicids2017_cleaned.csv")
OUTPUT_DIR = Path("../runs/processed")

DIST_DIR = OUTPUT_DIR / "distributions"
FULL_SPLIT_DIR = OUTPUT_DIR / "full_split"
SAMPLE_SPLIT_DIR = OUTPUT_DIR / "sample_split"

DIST_DIR.mkdir(parents=True, exist_ok=True)
FULL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)

print("Distribution dir:", DIST_DIR)
print("Full split dir:", FULL_SPLIT_DIR)
print("Sample split dir:", SAMPLE_SPLIT_DIR)

Data path: ..\datasets\cicids2017_cleaned.csv
Distribution dir: ..\runs\processed\distributions
Full split dir: ..\runs\processed\full_split
Sample split dir: ..\runs\processed\sample_split


In [3]:
#데이터 로드
df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
df.head()

Data shape: (2520751, 53)


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [4]:
#컬럼 확인
print("Columns:")
print(df.columns.tolist())

Columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Max', 'Active Min', 'Idle Mean', 'Idle 

In [5]:
#불필요 인덱스 컬럼 제거
unnamed_cols = [col for col in df.columns if col.startswith("Unnamed")]

if len(unnamed_cols) > 0:
    print("Remove columns:", unnamed_cols)
    df = df.drop(columns=unnamed_cols)

print("Data shape after column check:", df.shape)

Data shape after column check: (2520751, 53)


In [6]:
#원본 데이터 분포 확인
attack_type_counts = df["Attack Type"].value_counts()

print("Attack Type distribution:")
print(attack_type_counts)

Attack Type distribution:
Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64


In [7]:
#Binary Label 생성
df["Label"] = df["Attack Type"].apply(
    lambda x: 0 if x == "Normal Traffic" else 1
)

print("Binary label distribution:")
print(df["Label"].value_counts())

print("\nBinary label ratio:")
print(df["Label"].value_counts(normalize=True))

Binary label distribution:
Label
0    2095057
1     425694
Name: count, dtype: int64

Binary label ratio:
Label
0    0.831124
1    0.168876
Name: proportion, dtype: float64


In [8]:
#Feature / Label 분리
X = df.drop(columns=["Attack Type", "Label"])
y = df["Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2520751, 52)
y shape: (2520751,)


In [9]:
#잘못된 값 있는지 확인
missing_count = X.isna().sum().sum()

print("Missing values in X:", missing_count)

if missing_count > 0:
    raise ValueError("Feature에 결측치가 있습니다. cleaned dataset 상태를 다시 확인해야 합니다.")

Missing values in X: 0


In [10]:
# Step 1: train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Step 2: train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1765,
    random_state=42,
    stratify=y_train
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

#Split 비율 확인
total_size = len(X_train) + len(X_val) + len(X_test)

print("Split ratio")
print("Train ratio:     ", len(X_train) / total_size)
print("Validation ratio:", len(X_val) / total_size)
print("Test ratio:      ", len(X_test) / total_size)

#Label 비율 확인
print("Original label ratio")
print(y.value_counts(normalize=True))

print("\nTrain label ratio")
print(y_train.value_counts(normalize=True))

print("\nValidation label ratio")
print(y_val.value_counts(normalize=True))

print("\nTest label ratio")
print(y_test.value_counts(normalize=True))

#Split overlab 확인
train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

print("train-val overlap:", len(train_idx & val_idx))
print("train-test overlap:", len(train_idx & test_idx))
print("val-test overlap:", len(val_idx & test_idx))

Train: (1764462, 52) (1764462,)
Validation: (378176, 52) (378176,)
Test: (378113, 52) (378113,)
Split ratio
Train ratio:      0.6999747297531569
Validation ratio: 0.150025131399333
Test ratio:       0.15000013884751012
Original label ratio
Label
0    0.831124
1    0.168876
Name: proportion, dtype: float64

Train label ratio
Label
0    0.831124
1    0.168876
Name: proportion, dtype: float64

Validation label ratio
Label
0    0.831124
1    0.168876
Name: proportion, dtype: float64

Test label ratio
Label
0    0.831125
1    0.168875
Name: proportion, dtype: float64
train-val overlap: 0
train-test overlap: 0
val-test overlap: 0


In [11]:
# 기존 sample 파일 삭제
# train/validation/test sample을 새로 만들기 전에 이전 sample 파일을 제거한다.

sample_files = [
    "X_train_sample.csv",
    "X_val_sample.csv",
    "X_test_sample.csv",
    "y_train_sample.csv",
    "y_val_sample.csv",
    "y_test_sample.csv",
]

for file_name in sample_files:
    
    file_path = SAMPLE_SPLIT_DIR / file_name
    
    if file_path.exists():
        file_path.unlink()
        print("Deleted:", file_path)

print("Old sample files removed.")

Old sample files removed.


In [12]:
# KNN과 Random Forest의 공정한 sample 기반 비교를 위한 stratified sample 생성

TRAIN_SAMPLE_SIZE = 100_000
VAL_SAMPLE_SIZE = 30_000
TEST_SAMPLE_SIZE = 30_000

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLE_SIZE,
    random_state=42,
    stratify=y_train
)

X_val_sample, _, y_val_sample, _ = train_test_split(
    X_val,
    y_val,
    train_size=VAL_SAMPLE_SIZE,
    random_state=42,
    stratify=y_val
)

X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test,
    y_test,
    train_size=TEST_SAMPLE_SIZE,
    random_state=42,
    stratify=y_test
)

print("Train sample:", X_train_sample.shape, y_train_sample.shape)
print("Validation sample:", X_val_sample.shape, y_val_sample.shape)
print("Test sample:", X_test_sample.shape, y_test_sample.shape)

print("\nTrain sample label ratio")
print(y_train_sample.value_counts(normalize=True))

print("\nValidation sample label ratio")
print(y_val_sample.value_counts(normalize=True))

print("\nTest sample label ratio")
print(y_test_sample.value_counts(normalize=True))

Train sample: (100000, 52) (100000,)
Validation sample: (30000, 52) (30000,)
Test sample: (30000, 52) (30000,)

Train sample label ratio
Label
0    0.83112
1    0.16888
Name: proportion, dtype: float64

Validation sample label ratio
Label
0    0.831133
1    0.168867
Name: proportion, dtype: float64

Test sample label ratio
Label
0    0.831133
1    0.168867
Name: proportion, dtype: float64


In [13]:
#전체 Split 데이터 저장 - KNN/RF 파일에서 다시 Split 하지 않고 불러올 데이터
X_train.to_csv(FULL_SPLIT_DIR / "X_train.csv", index=False)
X_val.to_csv(FULL_SPLIT_DIR / "X_val.csv", index=False)
X_test.to_csv(FULL_SPLIT_DIR / "X_test.csv", index=False)

y_train.to_csv(FULL_SPLIT_DIR / "y_train.csv", index=False)
y_val.to_csv(FULL_SPLIT_DIR / "y_val.csv", index=False)
y_test.to_csv(FULL_SPLIT_DIR / "y_test.csv", index=False)

print("Full split data saved.")

Full split data saved.


In [14]:
# sample 데이터 저장

X_train_sample.to_csv(SAMPLE_SPLIT_DIR / "X_train_sample.csv", index=False)
X_val_sample.to_csv(SAMPLE_SPLIT_DIR / "X_val_sample.csv", index=False)
X_test_sample.to_csv(SAMPLE_SPLIT_DIR / "X_test_sample.csv" , index=False)

y_train_sample.to_csv(SAMPLE_SPLIT_DIR / "y_train_sample.csv", index=False)
y_val_sample.to_csv(SAMPLE_SPLIT_DIR / "y_val_sample.csv", index=False)
y_test_sample.to_csv(SAMPLE_SPLIT_DIR / "y_test_sample.csv", index=False)

print("Sample data saved.")

Sample data saved.


In [15]:
#데이터 개수 분포표 저장
binary_distribution = pd.DataFrame({
    "Class": ["Normal Traffic", "Attack Traffic"],
    "Label": [0, 1],
    "Count": [
        int((df["Label"] == 0).sum()),
        int((df["Label"] == 1).sum())
    ]
})

attack_type_distribution = df["Attack Type"].value_counts().reset_index()
attack_type_distribution.columns = ["Attack Type", "Count"]

binary_distribution.to_csv(DIST_DIR / "binary_label_distribution.csv", index=False)
attack_type_distribution.to_csv(DIST_DIR / "attack_type_distribution.csv", index=False)

print("Binary label distribution")
display(binary_distribution)

print("Attack type distribution")
display(attack_type_distribution)

Binary label distribution


,Class,Label,Count
0,Normal Traffic,0,2095057
1,Attack Traffic,1,425694


Attack type distribution


,Attack Type,Count
0,Normal Traffic,2095057
1,DoS,193745
2,DDoS,128014
3,Port Scanning,90694
4,Brute Force,9150
5,Web Attacks,2143
6,Bots,1948
